In [1]:
from __future__ import annotations

import csv
from pathlib import Path
from typing import Any, Iterable

import requests

In [ ]:
BASE_URL = "https://api.dados.pb.gov.br/api/v1"
CONTRATACOES_ENDPOINT = "api_dados/contratacoes"
MAX_PER_PAGE = 10


## Função Geral para coletar dados da API Compras

In [27]:
from __future__ import annotations

import csv
from pathlib import Path
from typing import Any, Iterable
from dataclasses import dataclass # Adicionado para importar dataclass

import requests


BASE_URL = "https://api.dados.pb.gov.br/api/v1"
CONTRATACOES_ENDPOINT = "/compras/contratacoes"
CONTRATOS_ENDPOINT  = "/compras/contratos"
ITENS_CONTRATACOES_ENDPOINT = "/compras/itenscontratacao"
MAX_PER_PAGE = 1000


class APIComprasPBError(RuntimeError):
    """Erro genérico de comunicação ou contrato de resposta da API."""
class APIComprasPBTimeout(APIComprasPBError):
    """Timeout ao consultar a API."""

# Define retorno para erros
class APIComprasPBError(RuntimeError):
    """Erro genérico de comunicação ou contrato de resposta da API."""
class APIComprasPBTimeout(APIComprasPBError):
    """Timeout ao consultar a API."""
@dataclass(frozen=True)

# Informações de paginação retornada pela API
class Pagination:
    """Informações de paginação retornadas pela API."""

    total_registros: int | None
    total_paginas: int | None
    pagina_atual: int | None
    registros_por_pagina: int | None
    raw: dict[str, Any]


    @classmethod
    def from_json(cls, payload: dict[str, Any] | None) -> "Pagination":
        if payload is None:
            return cls(None, None, None, None, {})
        if not isinstance(payload, dict):
            raise ValueError("A seção 'paginacao' deve ser um objeto JSON.")

        return cls(
            total_registros=payload.get("totalRegistros"),
            total_paginas=payload.get("totalPaginas"),
            pagina_atual=payload.get("paginaAtual"),
            registros_por_pagina=payload.get("registrosPorPagina"),
            raw=payload,
        )

# Resposta paginada
@dataclass(frozen=True)

class PaginatedResponse:
    """Resposta paginada padronizada pela API de Dados Abertos da Paraíba."""

    dados: list[dict[str, Any]]
    paginacao: Pagination
    raw: dict[str, Any]

    @classmethod
    def from_json(cls, payload: dict[str, Any]) -> "PaginatedResponse":
        if not isinstance(payload, dict):
            raise ValueError("A resposta da API deve ser um objeto JSON.")

        dados = payload.get("dados", [])
        if dados is None:
            dados = []
        if not isinstance(dados, list):
            raise ValueError("O campo 'dados' deve ser uma lista.")

        registros = []
        for item in dados:
            if not isinstance(item, dict):
                raise ValueError("Cada item em 'dados' deve ser um objeto JSON.")
            registros.append(item)

        return cls(
            dados=registros,
            paginacao=Pagination.from_json(payload.get("paginacao")),
            raw=payload,
        )

    @property
    def is_empty(self) -> bool:
        return len(self.dados) == 0

### API Compras Cliente -------------------------------------------------------
class APIComprasPBClient:
    """Cliente mínimo para exploração da API de compras da Paraíba."""

    def __init__(
        self,
        base_url: str = BASE_URL,
        timeout: float = 30.0,
        session: requests.Session | None = None,
    ) -> None:
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
        self.session = session or requests.Session()

    # Função para listar contratações  ----------------------------------

    def listar_contratacoes(
        self,
        ano: int,
        page: int = 1,
        per_page: int = 100,
        **filters: Any,
    ) -> PaginatedResponse:
        """Consulta uma página do endpoint /compras/contratos"""

        if per_page > MAX_PER_PAGE:
            raise ValueError(f"per_page não pode exceder {MAX_PER_PAGE}.")
        params = {"ano": ano, "page": page, "per_page": per_page}
        params.update({key: value for key, value in filters.items() if value not in (None, "")})
        return self._get_paginated(CONTRATACOES_ENDPOINT, params=params)

 ### Função para listar itens contratos  ----------------------------------

    def listar_contratos(
        self,
        anoInicioVigencia: int,
        page: int = 1,
        per_page: int = 100,
        **filters: Any,
    ) -> PaginatedResponse:
        """Consulta uma página do endpoint /compras/contratos."""

        if per_page > MAX_PER_PAGE:
            raise ValueError(f"per_page não pode exceder {MAX_PER_PAGE}.")
        params = {"anoInicioVigencia": anoInicioVigencia, "page": page, "per_page": per_page} # Corrigido aqui
        params.update({key: value for key, value in filters.items() if value not in (None, "")})
        return self._get_paginated(CONTRATOS_ENDPOINT, params=params)

### Função para iterar contratações ------------------------------------------------

    def iterar_contratacoes(
        self,
        ano: int,
        per_page: int = 1000,
        max_pages: int | None = None,
        **filters: Any,
    ) -> Iterable[dict[str, Any]]:
        """Itera por todas as páginas disponíveis, respeitando total_paginas."""

        page = 1
        while True:
            response = self.listar_contratacoes(
                ano=ano,
                page=page,
                per_page=per_page,
                **filters,
            )
            if response.is_empty:
                break
            yield from response.dados

            total_pages = response.paginacao.total_paginas
            if total_pages is not None and page >= total_pages:
                break
            if max_pages is not None and page >= max_pages:
                break
            page += 1

### Função para iterar contratos ------------------------------------------------

    def iterar_contratos(
        self,
        anoInicioVigencia: int,
        per_page: int = 1000,
        max_pages: int | None = None,
        **filters: Any,
    ) -> Iterable[dict[str, Any]]:
        """Itera por todas as páginas disponíveis, respeitando total_paginas."""

        page = 1
        while True:
            response = self.listar_contratos(
                anoInicioVigencia=anoInicioVigencia,
                page=page,
                per_page=per_page,
                **filters,
            )
            if response.is_empty:
                break
            yield from response.dados

            total_pages = response.paginacao.total_paginas
            if total_pages is not None and page >= total_pages:
                break
            if max_pages is not None and page >= max_pages:
                break
            page += 1


    def salvar_amostra_contratacoes(
        self,
        output_path: Path | str,
        ano: int,
        per_page: int = 100,
        page: int = 1,
        **filters: Any,
    ) -> PaginatedResponse:
        """Consulta uma página, normaliza os registros e salva uma amostra em CSV."""

        response = self.listar_contratacoes(
            ano=ano,
            page=page,
            per_page=per_page,
            **filters,
        )
        # Note: normalize_contratacao is not defined in the current context.
        # If you intend to use this method, you'll need to define it.
        rows = [item for item in response.dados] # Modified to avoid calling undefined normalize_contratacao
        path = Path(output_path)
        path.parent.mkdir(parents=True, exist_ok=True)

        if rows:
            fieldnames = list(rows[0].keys())
        else:
            fieldnames = []

        with path.open("w", newline="", encoding="utf-8") as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            if fieldnames:
                writer.writeheader()
                writer.writerows(rows)

        return response

    def _get_paginated(self, endpoint: str, params: dict[str, Any]) -> PaginatedResponse:
        url = f"{self.base_url}{endpoint}"
        try:
            response = self.session.get(url, params=params, timeout=self.timeout)
        except requests.Timeout as exc:
            raise APIComprasPBTimeout(f"Timeout ao consultar {url}") from exc
        except requests.RequestException as exc:
            raise APIComprasPBError(f"Falha ao consultar {url}: {exc}") from exc

        try:
            response.raise_for_status()
        except requests.HTTPError as exc:
            raise APIComprasPBError(
                f"Resposta HTTP inválida em {url}: {response.status_code} {response.text[:300]}"
            ) from exc

        try:
            payload = response.json()
        except ValueError as exc:
            raise APIComprasPBError(f"Resposta de {url} não é JSON válido.") from exc

        try:
            return PaginatedResponse.from_json(payload)
        except ValueError as exc:
            raise APIComprasPBError(f"Estrutura inesperada em {url}: {exc}") from exc

### 2. Iterar para pegar todas as contratações por ano. Ex: ano = 2025

In [19]:
import pandas as pd

# Re-instantiate the client (ensure it's available in this cell's context)
# If APIComprasPBClient definition is not in a preceding cell that is guaranteed to run,
# you might need to copy its definition here or ensure all definition cells are run.
# Assuming APIComprasPBClient is defined and accessible from cell 5hsw1_WM8wGy.
client = APIComprasPBClient()

print("\n--- Coletando todos os registros de contratações para 2025 ---")

all_contratacoes_2025 = []
try:
    for contratacao in client.iterar_contratacoes(ano=2025):
        all_contratacoes_2025.append(contratacao)

    print(f"Total de contratações coletadas para 2025: {len(all_contratacoes_2025)}")

    # Criar o DataFrame a partir da lista de dicionários
    df_contratacoes_2025 = pd.DataFrame(all_contratacoes_2025)

    print(f"O DataFrame resultante para 2025 tem {len(df_contratacoes_2025)} registros.")
    print("Exibindo as 5 primeiras linhas do DataFrame:")
    display(df_contratacoes_2025.head())

except APIComprasPBTimeout as e:
    print(f"Erro de timeout ao coletar dados: {e}")
except APIComprasPBError as e:
    print(f"Erro na API ao coletar dados: {e}")
except Exception as e:
    print(f"Ocorreu um erro inesperado ao coletar dados: {e}")


--- Coletando todos os registros de contratações para 2025 ---
Total de contratações coletadas para 2025: 3897
O DataFrame resultante para 2025 tem 3897 registros.
Exibindo as 5 primeiras linhas do DataFrame:


,numeroProcesso,numeroLicitacao,cadastroCge,tipoDocumento,nomeOrgao,objeto,modalidade,tipoLicitacao,criterioClassificacao,situacao,...,valorEstimado,valorAdjudicado,numParticipantes,registroPreco,amparoLegal,urlEdital,urlContratacao,urlPncp,documentos,participantes
0,26.201.031732.2025,006/2025,,TERMO DE REFERÊNCIA,SEDS/DEPARTAMENTO ESTADUAL DE TRÂNSITO DO ESTA...,CREDENCIAMENTO DE PESSOAS JURÍDICAS PARA FABRI...,CHAMAMENTO PÚBLICO,NÃO SE APLICA,VALOR GLOBAL,EM ANDAMENTO,...,0.00,0.0,NaN,N,"Lei 14.133/2021, Art. 51 e 74, V.",http://centraldecompras.pb.gov.br/appls/sgc/sg...,https://centraldecompras.pb.gov.br/appls/sgc/e...,,"[{""documento"":""TERMO DE REFERÊNCIA"",""url"":""htt...",[]
1,30.000.006605.2025,067/2025,,ATO QUE AUTORIZA A CONTRATAÇÃO DIRETA,ENCARGOS GERAIS DO ESTADO,AQUISIÇÃO DE EXTINTORES ORIUNDO DO CENTRO DE C...,DISPENSA DE LICITAÇÃO,MENOR PREÇO,VALOR GLOBAL,ENCERRADO,...,13880.00,13880.0,1.0,N,"Lei 14.133/2021, Art. 75, II.",https://centraldecompras.pb.gov.br/appls/sgc/s...,https://centraldecompras.pb.gov.br/appls/sgc/e...,https://pncp.gov.br/app/editais/08761140000356...,"[{""documento"":""ATO QUE AUTORIZA A CONTRATAÇÃO ...","[{""lote"":""N\/I"",""item"":0,""quantidade"":0.00,""cn..."
2,26.000.010084.2025,002/2025,25-02289-6,EDITAL,SECRETARIA DE ESTADO DA SEGURANÇA E DA DEFESA ...,CONTRATAÇÃO DE SOLUÇÃO TECNOLÓGICA COMPARTIMEN...,PREGÃO ELETRÔNICO,MENOR PREÇO,VALOR UNITÁRIO,ENCERRADO,...,223220.40,132480.0,1.0,N,"Lei 14.133/2021, Art. 28, I",https://centraldecompras.pb.gov.br/appls/sgc/s...,https://centraldecompras.pb.gov.br/appls/sgc/e...,https://pncp.gov.br/app/editais/08730095000100...,"[{""documento"":""EDITAL"",""url"":""http:\/\/central...","[{""lote"":""N\/I"",""item"":0,""quantidade"":0.00,""cn..."
3,27.000.002521.2025,001/2026,26-01729-6,EDITAL,SECRETARIA DE ESTADO DE DESENVOLVIMENTO HUMANO,CONSTRUÇÃO DO ESPAÇO ESPORTIVO COMUNITÁRIO NO ...,CONCORRÊNCIA - ELETRÔNICA,MENOR PREÇO,VALOR GLOBAL,EM ANDAMENTO,...,1453111.41,0.0,NaN,N,"Lei 14.133/2021, Art. 28, II",https://appcentral.centraldecompras.pb.gov.br/...,https://centraldecompras.pb.gov.br/appls/sgc/e...,https://pncp.gov.br/app/editais/08778276000107...,"[{""documento"":""EDITAL"",""url"":""https:\/\/appcen...",[]
4,25.000.045183.2025,018/2025,26-01587-0,EDITAL,SECRETARIA DE ESTADO DA SAÚDE,CONTRATAÇÃO DE EMPRESA PARA OBRA E REFORMA,CONCORRÊNCIA - ELETRÔNICA,MENOR PREÇO,VALOR UNITÁRIO,EM ANDAMENTO,...,2480849.30,0.0,NaN,N,"Lei 14.133/2021, Art. 28, II",https://centraldecompras.pb.gov.br/appls/sgc/s...,https://centraldecompras.pb.gov.br/appls/sgc/e...,https://pncp.gov.br/app/editais/08778268000160...,"[{""documento"":""AVISO DE LICITAÇÃO"",""url"":""http...",[]


## 3. Interar para pegar dados dos contratos por ano. Ex: ano = 2025

In [28]:
import pandas as pd

# Re-instantiate the client (ensure it's available in this cell's context)
# If APIComprasPBClient definition is not in a preceding cell that is guaranteed to run,
# you might need to copy its definition here or ensure all definition cells are run.
# Assuming APIComprasPBClient is defined and accessible from cell 5hsw1_WM8wGy.
client = APIComprasPBClient()

print("\n--- Coletando todos os registros de contratos para 2025 ---")

all_contratos_2025 = []
try:
    for contratos in client.iterar_contratos(anoInicioVigencia=2025):
        all_contratos_2025.append(contratos)

    print(f"Total de contratos coletadas para 2025: {len(all_contratos_2025)}")

    # Criar o DataFrame a partir da lista de dicionários
    df_contratos_2025 = pd.DataFrame(all_contratos_2025)

    print(f"O DataFrame resultante para 2025 tem {len(df_contratos_2025)} registros.")
    print("Exibindo as 5 primeiras linhas do DataFrame:")
    display(df_contratos_2025.head())

except APIComprasPBTimeout as e:
    print(f"Erro de timeout ao coletar dados: {e}")
except APIComprasPBError as e:
    print(f"Erro na API ao coletar dados: {e}")
except Exception as e:
    print(f"Ocorreu um erro inesperado ao coletar dados: {e}")


--- Coletando todos os registros de contratos para 2025 ---
Total de contratos coletadas para 2025: 2482
O DataFrame resultante para 2025 tem 2482 registros.
Exibindo as 5 primeiras linhas do DataFrame:


,anoInicioVigencia,registroCge,numeroContrato,numeroProcessoLicitacao,nomeOrgao,siglaOrgao,codigoOrgao,municipio,objeto,objetoComplemento,...,valorApostilas,valorTotal,emergencial,urlContrato,aditivo,nomeGestor,matriculaGestor,cpfGestor,dataPortaria,numeroPortariaGestor
0,2025,25-01873-6,0026/2025,34.204.842025.2025,COMPANHIA PARAIBANA DE GÁS,PBGÁS,34.0401,JOÃO PESSOA,OUTROS EVENTOS,"CONTRATAÇÃO DE SERVIÇOS DE LOCAÇÃO DE IMÓVEL, ...",...,0.0,611738.5,NÃO,https://cge.pb.gov.br/gea/Uploads/Contratos/Co...,"[{""motivo_aditivo"":""ALTERAR VALOR"",""num_aditiv...",ADRIANA DE FÁTIMA FERREIRA DO EGITO,006,***.006.928-**,2025-08-23,033/2025
1,2025,26-00384-8,0045/2025,21.201.000590.2025,CIA DE DESENVOLVIMENTO DA PARAÍBA,CINEP,21.0101,JOÃO PESSOA,EQUIPAMENTOS E SOFTWARES DE INFORMÁTICA,CONTRATAÇÃO DE EMPRESA PARA AQUISIÇÃO DE LICEN...,...,0.0,16985.0,NÃO,https://cge.pb.gov.br/gea/Uploads/Contratos/Co...,[],HERUNDINA KEYKHA CASTELO BRANCO,3210-1,***.005.431-**,2025-12-05,270/2025
2,2025,26-00341-4,0006/2025,,DEPARTAMENTO ESTADUAL DE TRÂNSITO DO ESTADO DA...,DETRAN,26.0101,JOÃO PESSOA,CONCESSÃO DE USO GRATUITO DE VEÍCULO,O PRESENTE TERMO TEM POR OBJETO A CESSÃO NÃO O...,...,0.0,0.0,NÃO,https://cge.pb.gov.br/gea/Uploads/Contratos/Co...,[],,,None,None,
3,2025,26-00190-0,0102/2025,,SUPERINTENDÊNCIA DA ADMINISTRAÇÃO DO MEIO AMBI...,SUDEMA,34.0101,JOÃO PESSOA,CESSÃO DE ESTÁGIO CURRICULAR,CESSÃO DE ESTÁGIO NÃO-OBRIGATÓRIO,...,0.0,9000.0,NÃO,https://cge.pb.gov.br/gea/Uploads/Contratos/Co...,[],DANILO AUGUSTO SANTOS DO NASCIMENTO,7206631,***.005.785-**,2025-12-16,SUDEMA/115/2025
4,2025,26-00155-1,0043/2025,19.204.000640.2025,CIA DE PROCESSAMENTOS DADOS DA PARAÍBA,CODATA,19.0401,JOÃO PESSOA,AQUISIÇÃO E INSTALAÇÃO DE MÁQUINAS E EQUIPAMENTOS,O OBJETO DO PRESENTE INSTRUMENTO É A AQUISIÇÃO...,...,0.0,411000.0,NÃO,https://cge.pb.gov.br/gea/Uploads/Contratos/Co...,[],RENANN BARBOSA MARTINS,7003439,***.009.983-**,2020-02-21,101


In [29]:
df_contratacoes_2025_agrupada = df_contratacoes_2025.groupby('amparoLegal').agg(
                                                                        {'valorEstimado':'sum','valorAdjudicado':'sum'})

In [30]:
df_contratacoes_2025_agrupada

,valorEstimado,valorAdjudicado
amparoLegal,,
Lei 13.303/2016 -14.133/2021,5.071656e+07,3.708488e+07
"Lei 13.303/2016, Art. 28",9.594581e+07,4.359473e+07
"Lei 13.303/2016, Art. 29, III",5.883454e+05,5.883454e+05
"Lei 14.133/2021, Art. 28, I",3.009486e+09,1.216701e+09
"Lei 14.133/2021, Art. 28, II",2.607000e+09,1.915683e+09
"Lei 14.133/2021, Art. 28, IV",1.997430e+06,0.000000e+00
"Lei 14.133/2021, Art. 51 e 74, V.",0.000000e+00,0.000000e+00
"Lei 14.133/2021, Art. 74, Caput",4.815575e+07,4.815575e+07
"Lei 14.133/2021, Art. 74, I",1.812533e+09,1.481413e+08


## 4. Todas as contratações

In [32]:
import pandas as pd

client = APIComprasPBClient()
ano = [2024, 2025, 2026]

print("\n--- Coletando todos os registros de contratações para os anos especificados ---")

all_contratacoes_list = []

try:
    for current_year in ano:
        print(f"Coletando contratações para o ano: {current_year}")
        contratacoes_for_year = []
        for contratacao in client.iterar_contratacoes(ano=current_year):
            # Adicionar a coluna de ano_referencia a cada dicionário de contratação
            contratacao['ano_referencia'] = current_year
            contratacoes_for_year.append(contratacao)

        print(f"Total de contratações coletadas para {current_year}: {len(contratacoes_for_year)}")
        all_contratacoes_list.extend(contratacoes_for_year)

    # Criar o DataFrame final a partir da lista combinada de dicionários
    df_contratacoes_todos_anos = pd.DataFrame(all_contratacoes_list)

    print(f"O DataFrame resultante para todos os anos tem {len(df_contratacoes_todos_anos)} registros.")
    print("Exibindo as 5 primeiras linhas do DataFrame consolidado:")
    display(df_contratacoes_todos_anos.head())
    print("Exibindo as últimas 5 linhas do DataFrame consolidado:")
    display(df_contratacoes_todos_anos.tail())
    print("Contagem de registros por ano_referencia:")
    display(df_contratacoes_todos_anos['ano_referencia'].value_counts())

except APIComprasPBTimeout as e:
    print(f"Erro de timeout ao coletar dados: {e}")
except APIComprasPBError as e:
    print(f"Erro na API ao coletar dados: {e}")
except Exception as e:
    print(f"Ocorreu um erro inesperado ao coletar dados: {e}")


--- Coletando todos os registros de contratações para os anos especificados ---
Coletando contratações para o ano: 2024
Total de contratações coletadas para 2024: 2288
Coletando contratações para o ano: 2025
Total de contratações coletadas para 2025: 3897
Coletando contratações para o ano: 2026
Total de contratações coletadas para 2026: 1890
O DataFrame resultante para todos os anos tem 8075 registros.
Exibindo as 5 primeiras linhas do DataFrame consolidado:


,numeroProcesso,numeroLicitacao,cadastroCge,tipoDocumento,nomeOrgao,objeto,modalidade,tipoLicitacao,criterioClassificacao,situacao,...,valorAdjudicado,numParticipantes,registroPreco,amparoLegal,urlEdital,urlContratacao,urlPncp,documentos,participantes,ano_referencia
0,19.000.000210.2024,079/2025,25-01141-8,EDITAL,SECRETARIA DE ESTADO DA ADMINISTRAÇÃO,REGISTRO DE PREÇOS PARA AQUISIÇÃO DE TOMÓGRAFO...,PREGÃO ELETRÔNICO,MENOR PREÇO,VALOR UNITÁRIO,EM ANDAMENTO,...,0.00,NaN,S,"Lei 14.133/2021, Art. 28, I",https://centraldecompras.pb.gov.br/appls/sgc/s...,https://centraldecompras.pb.gov.br/appls/sgc/e...,https://pncp.gov.br/app/editais/08761140000194...,"[{""documento"":""PUBLICIDADE DO SGC PREGÃO 079\/...",[],2024
1,30.000.005390.2024,040/2025,25-01730-5,EDITAL,ENCARGOS GERAIS DO ESTADO,CONTRATAÇÃO DE EMPRESA ESPECIALIZADA EM VIGILÂ...,PREGÃO ELETRÔNICO,MENOR PREÇO,VALOR GLOBAL,EM ANDAMENTO,...,0.00,NaN,N,"Lei 14.133/2021, Art. 28, I",https://centraldecompras.pb.gov.br/appls/sgc/s...,https://centraldecompras.pb.gov.br/appls/sgc/e...,https://pncp.gov.br/app/editais/08761140000356...,"[{""documento"":""EDITAL"",""url"":""http:\/\/central...",[],2024
2,19.000.000108.2024,096/2026,26-01552-2,EDITAL,SECRETARIA DE ESTADO DA ADMINISTRAÇÃO,REGISTRO DE PREÇOS PARA SERVIÇO DE LOCAÇÃO E M...,PREGÃO ELETRÔNICO,MENOR PREÇO,VALOR GLOBAL,EM ANDAMENTO,...,0.00,NaN,S,"Lei 14.133/2021, Art. 28, I",https://centraldecompras.pb.gov.br/appls/sgc/s...,https://centraldecompras.pb.gov.br/appls/sgc/e...,https://pncp.gov.br/app/editais/08761140000194...,"[{""documento"":""EDITAL"",""url"":""http:\/\/central...",[],2024
3,19.000.000213.2024,053/2025,25-01068-6,EDITAL,SECRETARIA DE ESTADO DA ADMINISTRAÇÃO,REGISTRO DE PREÇO PARA AQUISIÇÃO DE MONITORES ...,PREGÃO ELETRÔNICO,MENOR PREÇO,VALOR UNITÁRIO,EM ANDAMENTO,...,0.00,NaN,S,"Lei 14.133/2021, Art. 28, I",https://centraldecompras.pb.gov.br/appls/sgc/s...,https://centraldecompras.pb.gov.br/appls/sgc/e...,https://pncp.gov.br/app/editais/08761140000194...,"[{""documento"":""PUBLICIDADE DO 3º ADIAMENTO PRE...",[],2024
4,27.902.002856.2024,012/2026,26-00806-3,EDITAL,SEDH/FUNDO ESTADUAL DE ASSISTÊNCIA SOCIAL,AQUISIÇÃO DE MATERIAL DE LIMPEZA PARA O SERVIÇ...,PREGÃO ELETRÔNICO,MENOR PREÇO,VALOR UNITÁRIO,EM ANDAMENTO,...,25392.56,8.0,N,"Lei 14.133/2021, Art. 28, I",https://centraldecompras.pb.gov.br/appls/sgc/s...,https://centraldecompras.pb.gov.br/appls/sgc/e...,https://pncp.gov.br/app/editais/02467492000155...,"[{""documento"":""HABILITAÇÃO"",""url"":""http:\/\/ce...","[{""lote"":""N\/I"",""item"":0,""quantidade"":0.00,""cn...",2024


Exibindo as últimas 5 linhas do DataFrame consolidado:


,numeroProcesso,numeroLicitacao,cadastroCge,tipoDocumento,nomeOrgao,objeto,modalidade,tipoLicitacao,criterioClassificacao,situacao,...,valorAdjudicado,numParticipantes,registroPreco,amparoLegal,urlEdital,urlContratacao,urlPncp,documentos,participantes,ano_referencia
8070,09.901.000002.2026,002/2026,,ATO QUE AUTORIZA A CONTRATAÇÃO DIRETA,FUNDO ESTADUAL DE DEFESA DOS DIREITOS DO CONSU...,PASSAGENS AÉREAS,DISPENSA DE LICITAÇÃO,MENOR PREÇO,VALOR GLOBAL,ENCERRADO,...,60000.00,1.0,N,"Lei 14.133/2021, Art. 75, II.",https://centraldecompras.pb.gov.br/appls/sgc/s...,https://centraldecompras.pb.gov.br/appls/sgc/e...,https://pncp.gov.br/app/editais/21054904000170...,"[{""documento"":""ATO QUE AUTORIZA A CONTRATAÇÃO ...","[{""lote"":""N\/I"",""item"":0,""quantidade"":0.00,""cn...",2026
8071,09.901.000004.2026,004/2026,,ATO QUE AUTORIZA A CONTRATAÇÃO DIRETA,FUNDO ESTADUAL DE DEFESA DOS DIREITOS DO CONSU...,CONTRATAÇÃO DE EMPRESA PARA CONFECÇÃO DE CRACH...,DISPENSA DE LICITAÇÃO,MENOR PREÇO,VALOR UNITÁRIO,ENCERRADO,...,27100.00,1.0,N,"Lei 14.133/2021, Art. 75, II.",https://centraldecompras.pb.gov.br/appls/sgc/s...,https://centraldecompras.pb.gov.br/appls/sgc/e...,https://pncp.gov.br/app/editais/21054904000170...,"[{""documento"":""ATO QUE AUTORIZA A CONTRATAÇÃO ...","[{""lote"":""N\/I"",""item"":0,""quantidade"":0.00,""cn...",2026
8072,09.901.000005.2026,005/2026,,ATO QUE AUTORIZA A CONTRATAÇÃO DIRETA,FUNDO ESTADUAL DE DEFESA DOS DIREITOS DO CONSU...,CONTRATAÇÃO DE EMPRESA PARA CONFECÇÃO DE FARDA...,DISPENSA DE LICITAÇÃO,MENOR PREÇO,VALOR GLOBAL,ENCERRADO,...,36600.00,1.0,N,"Lei 14.133/2021, Art. 75, II.",https://centraldecompras.pb.gov.br/appls/sgc/s...,https://centraldecompras.pb.gov.br/appls/sgc/e...,https://pncp.gov.br/app/editais/21054904000170...,"[{""documento"":""ATO QUE AUTORIZA A CONTRATAÇÃO ...","[{""lote"":""N\/I"",""item"":0,""quantidade"":0.00,""cn...",2026
8073,35.204.013153.2026,039/2025,26-00364-5,EDITAL,SEE/UNIVERSIDADE ESTADUAL DA PARAÍBA,REGISTRO DE PREÇOS PARA AQUISIÇÃO DE MATERIAL ...,PREGÃO ELETRÔNICO,MENOR PREÇO,VALOR GLOBAL,ENCERRADO,...,392935.76,14.0,N,"Lei 14.133/2021, Art. 28, I",https://centraldecompras.pb.gov.br/appls/sgc/s...,https://centraldecompras.pb.gov.br/appls/sgc/e...,https://pncp.gov.br/app/editais/12671814000137...,"[{""documento"":""D.O.E - HOMOLOGAÇÃO"",""url"":""ht...","[{""lote"":""N\/I"",""item"":0,""quantidade"":0.00,""cn...",2026
8074,27.201.001441.2026,001/2026,,TERMO DE REFERÊNCIA,SEDH/FUNDAÇÃO DE DESENVOLVIMENTO DA CRIANÇA E ...,AQUISIÇÃO DE MATERIAL DE LIMPEZA,DISPENSA DE LICITAÇÃO,MENOR PREÇO,VALOR GLOBAL,EM ANDAMENTO,...,0.00,NaN,N,"Lei 14.133/2021, Art. 75, II.",http://centraldecompras.pb.gov.br/appls/sgc/sg...,https://centraldecompras.pb.gov.br/appls/sgc/e...,,"[{""documento"":""TERMO DE REFERÊNCIA"",""url"":""htt...",[],2026


Contagem de registros por ano_referencia:


,count
ano_referencia,
2025,3897
2024,2288
2026,1890


## 5. Todos os contratos (2024 a 2026)

In [34]:
import pandas as pd

client = APIComprasPBClient()
ano = [2024, 2025, 2026]

print("\n--- Coletando todos os registros de contratos anos especificados ---")

all_contratos_list = []

try:
    for current_year in ano:
        print(f"Coletando contratos para o ano: {current_year}")
        contratos_for_year = []
        for contrato in client.iterar_contratos(anoInicioVigencia=current_year):
            # Adicionar a coluna de ano_referencia a cada dicionário de contratos
            contrato['ano_referencia'] = current_year
            contratos_for_year.append(contrato)

        print(f"Total de contratos coletadas para {current_year}: {len(contratos_for_year)}")
        all_contratos_list.extend(contratos_for_year)

    # Criar o DataFrame final a partir da lista combinada de dicionários
    df_contratos_todos_anos = pd.DataFrame(all_contratos_list)

    print(f"O DataFrame resultante para todos os anos tem {len(df_contratos_todos_anos)} registros.")
    print("Exibindo as 5 primeiras linhas do DataFrame consolidado:")
    display(df_contratos_todos_anos.head())
    print("Exibindo as últimas 5 linhas do DataFrame consolidado:")
    display(df_contratos_todos_anos.tail())
    print("Contagem de registros por ano_referencia:")
    display(df_contratos_todos_anos['ano_referencia'].value_counts())

except APIComprasPBTimeout as e:
    print(f"Erro de timeout ao coletar dados: {e}")
except APIComprasPBError as e:
    print(f"Erro na API ao coletar dados: {e}")
except Exception as e:
    print(f"Ocorreu um erro inesperado ao coletar dados: {e}")


--- Coletando todos os registros de contratos anos especificados ---
Coletando contratos para o ano: 2024
Total de contratos coletadas para 2024: 4460
Coletando contratos para o ano: 2025
Total de contratos coletadas para 2025: 2482
Coletando contratos para o ano: 2026
Total de contratos coletadas para 2026: 1065
O DataFrame resultante para todos os anos tem 8007 registros.
Exibindo as 5 primeiras linhas do DataFrame consolidado:


,anoInicioVigencia,registroCge,numeroContrato,numeroProcessoLicitacao,nomeOrgao,siglaOrgao,codigoOrgao,municipio,objeto,objetoComplemento,...,valorTotal,emergencial,urlContrato,aditivo,nomeGestor,matriculaGestor,cpfGestor,dataPortaria,numeroPortariaGestor,ano_referencia
0,2024,24-12915-5,0547/2024,19.000.000079.2024,FUNDAÇÃO ESPAÇO CULTURAL DA PARAÍBA,FUNESC,33.0101,JOÃO PESSOA,O OBJETO DO PRESENTE INSTRUMENTO É A AQUISIÇÃO...,O OBJETO DO PRESENTE INSTRUMENTO É A AQUISIÇÃO...,...,18494.73,NÃO,https://pncp.gov.br/app/contratos/087611400001...,"[{""motivo_aditivo"":""ATUALIZAÇÃO RESERVA ORÇAME...",SEPHORA ARAUJO GOMES,175.488-2,***.004.745-**,2024-12-27,00252024,2024
1,2024,24-02598-4,0232/2024,31.206.015544.2023,COMPANHIA DE ÁGUA E ESGOTOS DO ESTADO DA PARAÍBA,CAGEPA,31.0601,JOÃO PESSOA,MATERIAL DE USO ESPECÍFICO,AQUISIÇÃO DE TUBOS PVC PARA REDE COLETORA DE E...,...,149999.40,NÃO,https://cge.pb.gov.br/gea/Uploads/Contratos/Co...,[],RENNYS DEMETRIUS DE LIMA FALCÃO,9327-3,***.000.768-**,2024-08-26,REDIR 034/17,2024
2,2024,24-02519-4,0767/2024,25.510.000394.2024,FUNDAÇÃO PARAIBANA DE GESTÃO EM SAÚDE,FPGS,25.5101,JOÃO PESSOA,MATERIAL MÉDICO-HOSPITALAR,AQUISIÇÃO DE MATERIAIS PARA A COMISSÃO DE PELE...,...,8076.90,NÃO,https://cge.pb.gov.br/gea/Uploads/Contratos/Co...,[],VÂNIA GOMES CABRAL,3987,***.002.178-**,2025-05-30,37,2024
3,2024,24-03002-3,0932/2024,25.510.000716.2024,FUNDAÇÃO PARAIBANA DE GESTÃO EM SAÚDE,FPGS,25.5101,JOÃO PESSOA,SERVIÇO ESPECIALIZADO,CONTRATAÇÃO DE SERVIÇO ESPECIALIZADO DE MONITO...,...,2100000.00,NÃO,https://cge.pb.gov.br/gea/Uploads/Contratos/Co...,"[{""motivo_aditivo"":""ALTERAR VIGÊNCIA"",""num_adi...",LOUISE NATHALIE QUEIROGA SEREJO FONTES,1880,***.001.970-**,2025-05-30,37,2024
4,2024,24-12296-0,0082/2024,15.000.000075.2024,POLICIA MILITAR DO ESTADO DA PARAIBA,PM/PB,15.0001,RECIFE,AQUISIÇÃO DE MATERIAL DE LIMPEZA (PAPEL HIGIÊN...,AQUISIÇÃO DE MATERIAL DE LIMPEZA (PAPEL HIGIÊN...,...,143105.00,NÃO,https://pncp.gov.br/app/contratos/089077760001...,[],SEVERINO FRANCISCO DA SILVA,527.083-9,***.006.930-**,2024-12-02,03112024,2024


Exibindo as últimas 5 linhas do DataFrame consolidado:


,anoInicioVigencia,registroCge,numeroContrato,numeroProcessoLicitacao,nomeOrgao,siglaOrgao,codigoOrgao,municipio,objeto,objetoComplemento,...,valorTotal,emergencial,urlContrato,aditivo,nomeGestor,matriculaGestor,cpfGestor,dataPortaria,numeroPortariaGestor,ano_referencia
8002,2026,26-00585-9,0009/2026,19.000.004225.2021,SECRETARIA DE ESTADO DA SAÚDE,SES,25.0001,JOÃO PESSOA,MATERIAL DE CONSUMO,"AQUISIÇÃO DE MATERIAL DE HIGIENE E LIMPEZA,CON...",...,6435.60,NÃO,https://cge.pb.gov.br/gea/Uploads/Contratos/Co...,[],SHIRLENE DANTAS GADELHA,925993,***.000.696-**,2019-11-21,0716/2019,2026
8003,2026,26-00051-2,0018/2025,,CONTROLADORIA GERAL DO ESTADO,CGE,11.0001,JOÃO PESSOA,BOLSISTA,TERMO DE COMPROMISSO Nº. 018/2025 PARA REALIZA...,...,18216.00,NÃO,https://cge.pb.gov.br/gea/Uploads/Contratos/Co...,[],MAYARA MARIA DE PONTES SILVA LIMA,1860364,***.008.452-**,2023-10-06,014/2023,2026
8004,2026,26-00053-9,0019/2025,,CONTROLADORIA GERAL DO ESTADO,CGE,11.0001,JOÃO PESSOA,BOLSISTA,TERMO DE COMPROMISSO Nº. 019/2025 PARA REALIZA...,...,18216.00,NÃO,https://cge.pb.gov.br/gea/Uploads/Contratos/Co...,[],MAYARA MARIA DE PONTES SILVA LIMA,1860364,***.008.452-**,2023-10-06,014/2023,2026
8005,2026,26-00078-4,0017/2025,,CONTROLADORIA GERAL DO ESTADO,CGE,11.0001,JOÃO PESSOA,BOLSISTA,TERMO DE COMPROMISSO Nº. 017/2025 PARA REALIZA...,...,18216.00,NÃO,https://cge.pb.gov.br/gea/Uploads/Contratos/Co...,[],MAYARA MARIA DE PONTES SILVA LIMA,1860364,***.008.452-**,2023-10-06,014/2023,2026
8006,2026,26-00331-7,1084/2026,,COMPANHIA ESTADUAL DE HABITAÇÃO POPULAR,CEHAP,31.0401,PRATA,CONSTRUÇÃO DE IMÓVEIS,O REFERIDO INSTRUMENTO JURIDICO FOI ANEXADO NE...,...,578026.24,NÃO,https://cge.pb.gov.br/gea/Uploads/Contratos/Co...,[],XXXXXXXXXX,0000000000,***.000.000-**,None,000000,2026


Contagem de registros por ano_referencia:


,count
ano_referencia,
2024,4460
2025,2482
2026,1065


## 3. Plano anual de contratações

In [40]:
js_plano_anual_contratacoes = !curl -X 'GET' \
  'https://api.dados.pb.gov.br/api/v1/compras/plano_anual_contratacao?ano=2026&page=1&per_page=200' \
  -H 'accept: application/json'

In [42]:
import pandas as pd
import json

# Ensure 'js' variable is available from the previous curl command
# Join the SList object into a single string
js_string = "\n".join(js_plano_anual_contratacoes)

# Parse the JSON string into a Python dictionary
js_dict = json.loads(js_string)

# Extract the 'dados' key, which contains the list of records
dados_list = js_dict.get('dados', [])

# Create a DataFrame from the list of records
js_df_plano_anual_contratacoes = pd.DataFrame(dados_list)

# Display the first few rows of the DataFrame
display(js_df_plano_anual_contratacoes.head())

,ano,cnpjOrgao,nomeOrgao,unidade,idItemPca,categoriaItem,classeGrupo,codigoItem,descricaoItem,unidadeFornecimento,quantidade,valorUnitario,valorTotal,valorOrcamentoExercicio,dataDesejada,linkPca,linkItemPca
0,2026,02102173000146,FUNDO ESPECIAL DE DESENV DE RECURSOS HUMANOS,19901-FUNDO ESPECIAL DE DESENVOLVIMENTO DE REC...,1,MATERIAL,33903016-MATERIAL DE EXPEDIENTE,45290,"AGENDA tipo diária, ano seguinte, dimensões (2...",UN,50.0,32.0,1600.0,1600.0,2027-12-31,https://pncp.gov.br/app/pca/02102173000146/2026,https://pncp.gov.br/app/pca/02102173000146/2026/1
1,2026,02221962000104,SECRETARIA DE ESTADO DA INFRAESTRUTURA DOS REC...,31103-SEC DE ESTADO DA INFRAESTRUTURA E DOS RE...,1,MATERIAL,33903004-GAS ENGARRAFADO,72932,"RECARGA de extintores, conforme detalhamento e...",UN,8.0,1000.0,8000.0,8000.0,2027-12-31,https://pncp.gov.br/app/pca/02221962000104/2026,https://pncp.gov.br/app/pca/02221962000104/2026/1
2,2026,02467492000155,FUNDO ESTADUAL DE ASSISTENCIA SOCIAL,27902-FUNDO ESTADUAL DE ASSISTENCIA SOCIAL (FEAS),1,SERVIÇO,33903699-OUTROS SERVICOS DE PESSOA FÍSICA,92162,CONTRATAÇÃO de pessoa física para prestação de...,UN,5.0,2710.0,13550.0,13550.0,2027-12-31,https://pncp.gov.br/app/pca/02467492000155/2026,https://pncp.gov.br/app/pca/02467492000155/2026/1
3,2026,03114093000173,SECRETARIA DE ESTADO DA COMUNICACAO INSTITUCIONAL,29101-SECRETARIA DE ESTADO DA COMUNICACAO INST...,1,MATERIAL,"44905234-MAQUINAS, UTENSILIOS E EQUIPAMENTOS ...",72332,CONDICIONADOR de ar tipo Split Hi Wall inverte...,UN,2.0,3500.0,7000.0,7000.0,2027-12-31,https://pncp.gov.br/app/pca/03114093000173/2026,https://pncp.gov.br/app/pca/03114093000173/2026/1
4,2026,04838295000120,AGENCIA DE REGULACAO DO ESTADO DA PARAIBA,9202-AGÊNCIA DE REGULACÃO DO ESTADO DA PARAÍBA...,1,MATERIAL,33903099-OUTROS MATERIAIS DE CONSUMO,36179,"ALFINETE para quadro de cortiça tipo taça, com...",EMB,5.0,7.0,35.0,35.0,2027-12-31,https://pncp.gov.br/app/pca/04838295000120/2026,https://pncp.gov.br/app/pca/04838295000120/2026/1


In [44]:
import requests
import pandas as pd
import json

# Define o URL base para o endpoint 'compras/contratos'
CONTRACTS_BASE_URL_PLANO = 'https://api.dados.pb.gov.br/api/v1/compras/plano_anual_contratacao'

# Define o ano para buscar os dados
target_year = 2026 # O usuário especificamente pediu para 2025

# Inicializa uma lista vazia para armazenar todos os dados coletados
all_contracts_data_plano = []

# Define os parâmetros iniciais de paginação
page = 1
per_page = 1000 # Maximize per_page para reduzir o número de requisições

print(f"--- Coletando todos os registros de contratos para o ano: {target_year} ---")

# Usa uma sessão para melhor desempenho com múltiplas requisições
with requests.Session() as session:
    while True:
        params = {
            'ano': target_year,
            'page': page,
            'per_page': per_page
        }
        headers = {
            'accept': 'application/json'
        }

        try:
            response = session.get(CONTRACTS_BASE_URL_PLANO, params=params, headers=headers, timeout=30)
            response.raise_for_status() # Levanta um HTTPError para respostas de erro (4xx ou 5xx)
            data = response.json()

            # Verifica se a chave 'dados' existe e é uma lista
            if 'dados' in data and isinstance(data['dados'], list):
                current_page_dados = data['dados']

                if not current_page_dados:
                    # Não há mais dados nesta página ou a última página estava vazia
                    print(f"Nenhum dado encontrado na página {page}, finalizando coleta.")
                    break

                # Adiciona 'ano_referencia' a cada registro antes de estender a lista
                for record in current_page_dados:
                    record['ano_referencia'] = target_year

                all_contracts_data_plano.extend(current_page_dados)
                print(f"Página {page} coletada: {len(current_page_dados)} registros.")

                # Verifica as informações de paginação para decidir se continua
                pagination = data.get('paginacao', {})
                total_pages = pagination.get('totalPaginas')

                if total_pages is not None and page >= total_pages:
                    print(f"Total de páginas ({total_pages}) atingido.")
                    break

                page += 1
            else:
                print(f"Resposta inesperada da API na página {page}: 'dados' não encontrado ou não é uma lista.")
                print(f"Resposta bruta: {data}")
                break # Quebra em caso de estrutura de resposta inesperada

        except requests.exceptions.Timeout as e:
            print(f"Erro de timeout ao coletar dados da página {page}: {e}")
            break
        except requests.exceptions.RequestException as e:
            print(f"Erro ao fazer requisição HTTP para a página {page}: {e}")
            break
        except json.JSONDecodeError as e:
            print(f"Erro ao decodificar JSON da resposta da página {page}: {e}")
            if 'response' in locals():
                print(f"Resposta de texto bruta: {response.text[:500]}...") # Limita o texto para exibição
            break
        except Exception as e:
            print(f"Ocorreu um erro inesperado na página {page}: {e}")
            break

print(f"\nTotal de registros coletados para {target_year}: {len(all_contracts_data)}")

# Cria um DataFrame a partir dos dados coletados
if all_contracts_data_plano:
    df_contratos_data_plano_todos = pd.DataFrame(all_contracts_data_plano)
    print(f"DataFrame 'df_contratos e planos' criado com {len(df_contratos_data_plano_todos)} registros.")
    print("Exibindo as 5 primeiras linhas do DataFrame:")
    display(df_contratos_data_plano_todos.head())
    print("\nVerificando a coluna 'ano_referencia':")
    display(df_contratos_data_plano_todos['ano'].value_counts())
else:
    print("Nenhum dado foi coletado para criar o DataFrame.")

--- Coletando todos os registros de contratos para o ano: 2026 ---
Página 1 coletada: 1000 registros.
Página 2 coletada: 1000 registros.
Página 3 coletada: 1000 registros.
Página 4 coletada: 1000 registros.
Página 5 coletada: 1000 registros.
Página 6 coletada: 1000 registros.
Página 7 coletada: 1000 registros.
Página 8 coletada: 1000 registros.
Página 9 coletada: 1000 registros.
Página 10 coletada: 1000 registros.
Página 11 coletada: 1000 registros.
Página 12 coletada: 1000 registros.
Página 13 coletada: 1000 registros.
Página 14 coletada: 1000 registros.
Página 15 coletada: 1000 registros.
Página 16 coletada: 1000 registros.
Página 17 coletada: 69 registros.
Nenhum dado encontrado na página 18, finalizando coleta.

Total de registros coletados para 2026: 1065
DataFrame 'df_contratos e planos' criado com 16069 registros.
Exibindo as 5 primeiras linhas do DataFrame:


,ano,cnpjOrgao,nomeOrgao,unidade,idItemPca,categoriaItem,classeGrupo,codigoItem,descricaoItem,unidadeFornecimento,quantidade,valorUnitario,valorTotal,valorOrcamentoExercicio,dataDesejada,linkPca,linkItemPca,ano_referencia
0,2026,02102173000146,FUNDO ESPECIAL DE DESENV DE RECURSOS HUMANOS,19901-FUNDO ESPECIAL DE DESENVOLVIMENTO DE REC...,1,MATERIAL,33903016-MATERIAL DE EXPEDIENTE,45290,"AGENDA tipo diária, ano seguinte, dimensões (2...",UN,50.0,32.0,1600.0,1600.0,2027-12-31,https://pncp.gov.br/app/pca/02102173000146/2026,https://pncp.gov.br/app/pca/02102173000146/2026/1,2026
1,2026,02221962000104,SECRETARIA DE ESTADO DA INFRAESTRUTURA DOS REC...,31103-SEC DE ESTADO DA INFRAESTRUTURA E DOS RE...,1,MATERIAL,33903004-GAS ENGARRAFADO,72932,"RECARGA de extintores, conforme detalhamento e...",UN,8.0,1000.0,8000.0,8000.0,2027-12-31,https://pncp.gov.br/app/pca/02221962000104/2026,https://pncp.gov.br/app/pca/02221962000104/2026/1,2026
2,2026,02467492000155,FUNDO ESTADUAL DE ASSISTENCIA SOCIAL,27902-FUNDO ESTADUAL DE ASSISTENCIA SOCIAL (FEAS),1,SERVIÇO,33903699-OUTROS SERVICOS DE PESSOA FÍSICA,92162,CONTRATAÇÃO de pessoa física para prestação de...,UN,5.0,2710.0,13550.0,13550.0,2027-12-31,https://pncp.gov.br/app/pca/02467492000155/2026,https://pncp.gov.br/app/pca/02467492000155/2026/1,2026
3,2026,03114093000173,SECRETARIA DE ESTADO DA COMUNICACAO INSTITUCIONAL,29101-SECRETARIA DE ESTADO DA COMUNICACAO INST...,1,MATERIAL,"44905234-MAQUINAS, UTENSILIOS E EQUIPAMENTOS ...",72332,CONDICIONADOR de ar tipo Split Hi Wall inverte...,UN,2.0,3500.0,7000.0,7000.0,2027-12-31,https://pncp.gov.br/app/pca/03114093000173/2026,https://pncp.gov.br/app/pca/03114093000173/2026/1,2026
4,2026,04838295000120,AGENCIA DE REGULACAO DO ESTADO DA PARAIBA,9202-AGÊNCIA DE REGULACÃO DO ESTADO DA PARAÍBA...,1,MATERIAL,33903099-OUTROS MATERIAIS DE CONSUMO,36179,"ALFINETE para quadro de cortiça tipo taça, com...",EMB,5.0,7.0,35.0,35.0,2027-12-31,https://pncp.gov.br/app/pca/04838295000120/2026,https://pncp.gov.br/app/pca/04838295000120/2026/1,2026



Verificando a coluna 'ano_referencia':


,count
ano,
2026,16069
